# 40.14 Согласованный КТ/FEM-анализ чувствительности ТТРКГ и боковых сборок — реокардиомонитор РНЦХ

**Статус:** протокол расчёта подготовлен; численные FEM-результаты отсутствуют
до успешного запуска MATLAB/EIDORS и принятия монтажа `40.02–40.03`.

## Задача

Одни и те же изменения параметров индивидуальной КТ-модели оцениваются для
ТТРКГ и боковых сборок 140 мм. Простого сходства
карт недостаточно: нужны численные производные, знак, масштаб, устойчивость к
шагу возмущения и обусловленность общей матрицы.

Для РНЦХ сначала рассчитывается физический монтаж канала 1 при отключённом канале 2. Результат `40.12` нельзя автоматически превратить в поправку FEM: он описывает приборный эффект, но не устанавливает его механизм и переносимость между записями.


## Математическое определение

Для параметра (p) и выхода (Y)

\[
\frac{\partial Y}{\partial p}\approx
\frac{Y(p_0+\Delta p)-Y(p_0-\Delta p)}{2\Delta p},\qquad
S_p^Y=\frac{p_0}{Y_{scale}}\frac{\partial Y}{\partial p}.
\]

Для базового импеданса (Y_{scale}=|Z_0|). Для знакопеременной кривой
(\Delta Z(t)) используется один фиксированный положительный масштаб базовой
кривой (RMS или `q99−q01`), а не поточечное деление на значения около нуля.

Шаги 1, 2 и 5 % (или 1, 2 и 5 мм для положения электрода) — вычислительная
сетка исследования производной, не физиологический диапазон неопределённости.


In [1]:
from __future__ import annotations
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from ttrkg_analysis import relative_derivative_disagreement, scaled_local_sensitivity

EXPERIMENT_ID = "exp03"
NOTEBOOK_NUMBER = "40.14"
PROTOCOL_VERSION = "40_fem_sensitivity_v1"
REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"

protocol = {
    "experiment_id": EXPERIMENT_ID,
    "protocol_version": PROTOCOL_VERSION,
    "prerequisites": ["40.02 geometry accepted", "40.02 reciprocity accepted", "manual Inobitec segmentation hashes recorded", "mesh convergence assessed separately"],
    "measurement_families": {"ttrkg": {"electrode_types": ["top_disc_5mm", "circumferential_ring_width_5mm", "area_equivalent_surface_cuff"], "position_parameters_mm": ["d_in", "d_out"]}, "lateral": {"side_sizes_mm": [140], "position_source": "individual_CT_FEM_localization"}},
    "continuous_parameters": [
        {"parameter": "rho_soft_tissue", "units": "ohm_m", "steps_relative": [0.01, 0.02, 0.05]},
        {"parameter": "rho_lung", "units": "ohm_m", "steps_relative": [0.01, 0.02, 0.05]},
        {"parameter": "rho_heart_muscle", "units": "ohm_m", "steps_relative": [0.01, 0.02, 0.05]},
        {"parameter": "rho_blood", "units": "ohm_m", "steps_relative": [0.01, 0.02, 0.05], "status": "requires_blood_mask_or_explicit_volume_scenario"},
        {"parameter": "rho_bone", "units": "ohm_m", "steps_relative": [0.01, 0.02, 0.05]},
        {"parameter": "contact_impedance", "units": "ohm_m2", "steps_relative": [0.01, 0.02, 0.05]},
        {"parameter": "d_in", "units": "mm", "steps_absolute": [1.0, 2.0, 5.0]},
        {"parameter": "d_out", "units": "mm", "steps_absolute": [1.0, 2.0, 5.0]},
    ],
    "categorical_scenarios": ["electrode_type", "blood_mask_absent_or_present", "respiratory_geometry"],
    "outputs": [{"output": "base_transfer_impedance", "units": "ohm"}, {"output": "regional_dZ_dlnrho", "units": "ohm"}, {"output": "delta_z_waveform", "units": "ohm"}],
    "required_diagnostics": ["signed derivative", "scaled sensitivity", "step disagreement", "reciprocity residual", "mesh convergence", "joint matrix rank and condition"],
}
display(pd.DataFrame(protocol["continuous_parameters"]))
print(json.dumps(protocol["measurement_families"], ensure_ascii=False, indent=2))


,parameter,units,steps_relative,status,steps_absolute
0,rho_soft_tissue,ohm_m,"[0.01, 0.02, 0.05]",NaN,NaN
1,rho_lung,ohm_m,"[0.01, 0.02, 0.05]",NaN,NaN
2,rho_heart_muscle,ohm_m,"[0.01, 0.02, 0.05]",NaN,NaN
3,rho_blood,ohm_m,"[0.01, 0.02, 0.05]",requires_blood_mask_or_explicit_volume_scenario,NaN
4,rho_bone,ohm_m,"[0.01, 0.02, 0.05]",NaN,NaN
5,contact_impedance,ohm_m2,"[0.01, 0.02, 0.05]",NaN,NaN
6,d_in,mm,NaN,NaN,"[1.0, 2.0, 5.0]"
7,d_out,mm,NaN,NaN,"[1.0, 2.0, 5.0]"


{
  "ttrkg": {
    "electrode_types": [
      "top_disc_5mm",
      "circumferential_ring_width_5mm",
      "area_equivalent_surface_cuff"
    ],
    "position_parameters_mm": [
      "d_in",
      "d_out"
    ]
  },
  "lateral": {
    "side_sizes_mm": [
      140
    ],
    "position_source": "individual_CT_FEM_localization"
  }
}


In [2]:
p0, step = 3.0, 0.03
check = scaled_local_sensitivity((p0 - step) ** 2, p0**2, (p0 + step) ** 2, p0, step)
assert abs(float(check["scaled_sensitivity"]) - 2.0) < 1e-12
small = scaled_local_sensitivity([0.99, 1.98], [1.0, 2.0], [1.01, 2.02], 1.0, 0.01, output_scale=2.0)
large = scaled_local_sensitivity([0.95, 1.90], [1.0, 2.0], [1.05, 2.10], 1.0, 0.05, output_scale=2.0)
assert relative_derivative_disagreement(small["derivative"], large["derivative"]) < 1e-12
print(f"{NOTEBOOK_NUMBER} sensitivity_math_self_test: passed")


40.14 sensitivity_math_self_test: passed


In [3]:
REQUIRED_ENTRY_FIELDS = {"measurement_id", "montage_family", "parameter_id", "parameter_zero", "step", "output_id", "output_scale", "y_minus", "y_zero", "y_plus"}
if not REAL_MODE:
    print(f"{NOTEBOOK_NUMBER} FEM status: pending_matlab_eidors_run")
else:
    index_path = Path(os.environ["KALMYKOV_FEM_SENSITIVITY_INDEX"]).expanduser().resolve()
    index = json.loads(index_path.read_text(encoding="utf-8"))
    if index.get("status") != "accepted" or index.get("experiment_id") != EXPERIMENT_ID or index.get("protocol_version") != PROTOCOL_VERSION:
        raise RuntimeError("Индекс FEM не принят или не совпал с протоколом")
    setup = index.get("electrode_setup_test", {})
    if setup.get("status") != "accepted" or setup.get("reciprocity_passed") is not True:
        raise RuntimeError("Монтаж и взаимность 40.02 не приняты")
    if index.get("mesh_convergence", {}).get("status") != "accepted":
        raise RuntimeError("Не принята сеточная сходимость")
    entries = index.get("entries", [])
    rows, derivatives = [], {}
    for entry in entries:
        if not REQUIRED_ENTRY_FIELDS.issubset(entry):
            raise RuntimeError("Неполная запись FEM-чувствительности")
        result = scaled_local_sensitivity(entry["y_minus"], entry["y_zero"], entry["y_plus"], entry["parameter_zero"], entry["step"], output_scale=entry["output_scale"])
        key = (entry["measurement_id"], entry["parameter_id"], entry["output_id"])
        derivatives.setdefault(key, []).append((float(entry["step"]), np.asarray(result["derivative"])))
        rows.append({"measurement_id": entry["measurement_id"], "montage_family": entry["montage_family"], "parameter_id": entry["parameter_id"], "output_id": entry["output_id"], "step": entry["step"], "scaled_sensitivity_l2": float(np.linalg.norm(result["scaled_sensitivity"]))})
    stability = []
    for key, estimates in derivatives.items():
        estimates = sorted(estimates, key=lambda item: item[0])
        for first, second in zip(estimates, estimates[1:]):
            stability.append({"measurement_id": key[0], "parameter_id": key[1], "output_id": key[2], "step_1": first[0], "step_2": second[0], "relative_derivative_disagreement": relative_derivative_disagreement(first[1], second[1])})
    display(pd.DataFrame(rows))
    display(pd.DataFrame(stability))


40.14 FEM status: pending_matlab_eidors_run


## Что считается результатом

Результатом будет таблица производных и матрица «монтаж × параметр» с
прослеживаемыми входами. Карты служат пространственным объяснением чисел.
Физическая чувствительность, категориальные различия формы контакта, численная
ошибка сетки и неопределённость эксперимента учитываются раздельно.

Совпадение базового импеданса, взаимность и устойчивость к шагу необходимы, но
по отдельности не доказывают анатомическую правильность региональной карты.
Пока MATLAB/EIDORS не выполнит серию, научного численного вывода здесь нет.
